In [12]:
%load_ext autoreload
%autoreload 2

import generate_map
generate_map.build_map()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


c:\Users\HP\Desktop\texel_map\generate_map.py:504: UserWarning: color argument of Icon should be one of: {'gray', 'lightred', 'pink', 'purple', 'white', 'darkpurple', 'beige', 'orange', 'lightblue', 'black', 'red', 'green', 'lightgreen', 'darkgreen', 'lightgray', 'cadetblue', 'darkred', 'blue', 'darkblue'}.


Successfully generated texel_map.html!


In [6]:
import qrcode
img = qrcode.make('https://hamin-yoon.github.io/Texel_map/texel_map.html')
img.save("texel_map_qr.png")

In [9]:
import folium
from folium import FeatureGroup, LayerControl, Element
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import gpxpy
import os
import gpxpy.gpx
import webbrowser  # Add this at the top of generate_map.py
import json
def geocode_addresses(df):
    """Fills in missing latitude and longitude using geopy."""
    geolocator = Nominatim(user_agent="texel_map_builder")
    # Rate limiter ensures we respect OpenStreetMap free service limits (1 sec delay)
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=0.1)
    
    updated = False
    for index, row in df.iterrows():
        # Only geocode if latitude/longitude are missing
        if pd.isna(row['latitude']) or pd.isna(row['longitude']):
            address = row['address']
            print(f"Geocoding address: {address}...")
            try:
                location = geocode(address)
                if location:
                    df.at[index, 'latitude'] = location.latitude
                    df.at[index, 'longitude'] = location.longitude
                    updated = True
                    print(f"  -> Found: {location.latitude}, {location.longitude}")
                else:
                    print(f"  -> Could not find coordinates for: {address}")
            except Exception as e:
                print(f"  -> Error geocoding {address}: {e}")
            time.sleep(1)

    # Save the updated coordinates back to CSV so we don't re-fetch them every time
    if updated:
        df.to_csv("places.csv", index=False)
        print("Updated places.csv with new coordinates!")

    return df
sheet_url = "https://docs.google.com/spreadsheets/d/1qEE9cMS7sf_Apa-tP8KOQt81zJhOpeu_L2RxPYZov1Q/export?format=csv&gid=716767332"
df = pd.read_csv(sheet_url)

# 1b. Automatically fill in missing columns with empty data so the script doesn't crash
missing_columns = ['latitude', 'longitude', 'description_nl', 'description_en', 'name_nl', 'name_en']
for col in missing_columns:
    if col not in df.columns:
        df[col] = pd.NA
        
# 1c. Fallback: Use the basic 'name' column for both NL and EN popups if they are empty
if 'name' in df.columns:
    df['name_nl'] = df['name_nl'].fillna(df['name'])
    df['name_en'] = df['name_en'].fillna(df['name'])

# The geocoder will now see the empty 'latitude'/'longitude' columns and fill them!
df = geocode_addresses(df)

# 2. Base Map centered on Texel
texel_map = folium.Map(
    location=[53.0583, 4.8018], 
    zoom_start=11, 
    tiles="CartoDB positron"
)

    # 4. Create Dynamic Categories (Feature Groups) FIRST
categories = df['category'].dropna().unique()
feature_groups = {}

for cat in categories:
    fg = FeatureGroup(name=cat)
    fg.add_to(texel_map)
    feature_groups[cat] = fg

texel_map = folium.Map(
    location=[53.0583, 4.8018], 
    zoom_start=11, 
    tiles="CartoDB positron"
)

gpx_folder = "gpx"
if os.path.exists(gpx_folder):
    for filename in sorted(os.listdir(gpx_folder)):
        if filename.lower().endswith(".gpx"):
            # Formats filename (e.g. 'gallery_route_1.gpx' -> 'Gallery Route 1')
            route_name = os.path.splitext(filename)[0].replace("_", " ").title()
            
            gpx_path = os.path.join(gpx_folder, filename)
            
            try:
                with open(gpx_path, 'r', encoding='utf-8') as gpx_file:
                    gpx = gpxpy.parse(gpx_file)

                points = []
                for track in gpx.tracks:
                    for segment in track.segments:
                        for point in segment.points:
                            points.append((point.latitude, point.longitude))

                if points:
                    route_fg = FeatureGroup(name=route_name)
                    route_fg.add_to(texel_map)
                    print(f"{route_name}  {(points)} points.")

                    folium.PolyLine(
                        locations=points,
                        color="blue",
                        weight=5,
                        opacity=0.7,
                        tooltip=route_name
                    ).add_to(route_fg)

                    # Registers layer so UI toggle buttons generate automatically
                    feature_groups[route_name] = route_fg
                else:
                    print(route_name)    
            except Exception as e:
                print(f"Error parsing GPX file {filename}: {e}")

Boeten
Cultuurhistorische Route-Linestring  [(53.11782, 4.8276), (53.11796, 4.82779), (53.11798, 4.82775), (53.11804, 4.82765), (53.11826, 4.82716), (53.11876, 4.82776), (53.11878, 4.82776), (53.11879, 4.82775), (53.11887, 4.82755), (53.11921, 4.82796), (53.11948, 4.82825), (53.12153, 4.83063), (53.12158, 4.83068), (53.12361, 4.83303), (53.12368, 4.83318), (53.12452, 4.83419), (53.12456, 4.8342), (53.1246, 4.8342), (53.13449, 4.8459), (53.13465, 4.8461), (53.13552, 4.84712), (53.13555, 4.84716), (53.13558, 4.84721), (53.13535, 4.84759), (53.13534, 4.8477), (53.13524, 4.8479), (53.1352, 4.84796), (53.13511, 4.84819), (53.13508, 4.84825), (53.12899, 4.86245), (53.12894, 4.86255), (53.12888, 4.86262), (53.12872, 4.86294), (53.1287, 4.86307), (53.12869, 4.8632), (53.12744, 4.86613), (53.12743, 4.86617), (53.12073, 4.88185), (53.12069, 4.88193), (53.12061, 4.88216), (53.11924, 4.88151), (53.1192, 4.8815), (53.11918, 4.88151), (53.11915, 4.88151), (53.11912, 4.88153), (53.1191, 4.88157), (53

In [1]:
import pandas as pd
pd.read_csv('places.csv')

,category,name_nl,name_en,address,website,description_nl,description_en,latitude,longitude
0,Galerieën,Atelier Galerie Texel Kunst - Angelina Stiehl,Atelier Galerie Texel Kunst - Angelina Stiehl,"Sluyscoog 79, 1791 WV Den Burg, Texel",https://www.texel.net/,Atelier en galerie van Angelina Stiehl.,Studio and gallery of Angelina Stiehl.,53.055653,4.806063
1,Galerieën,De Eiland Galerij,De Eiland Galerij,"Postweg 72, 1795 JR De Cocksdorp, Texel",https://www.texel.net/nl/activiteit/keramiek-i...,Expositie keramiek in de beeldentuin.,Ceramics exhibition in the sculpture garden.,53.112013,4.819997
2,Galerieën,Schapenboet Wieg Kunst,Schapenboet Cradle Art,"Stolpweg 20a, 1797 AV Den Hoorn, Texel",https://www.texel.net/nl/activiteit/wieg-kunst...,Wieg kunst expositie in traditionele schapenboet.,Cradle art exhibition in a traditional sheep b...,53.020062,4.755799
3,Galerieën,Wolkathedraal,Wool Cathedral,"Peperstraat 7, 1793 AA Oosterend, Texel",https://www.texel.net/nl/activiteit/wolkathedr...,Bijzondere kunstexpositie van wol in Oosterend.,Unique wool art exhibition in Oosterend.,53.085014,4.873689
4,Galerieën,Expositie Texels Landschap,Texel Landscape Exhibition,"Brink 14, 1796 AJ De Koog, Texel",https://www.texel.net/nl/activiteit/posthuys-e...,Expositie van het Texelse landschap bij Galeri...,Exhibition of the Texel landscape at Galerie P...,53.097239,4.761309
5,Gallery-route1,01 QUUB,01 QUUB,"Kogerweg 145, 1791 MJ Den Burg, Texel",https://www.quubkunst.nl,"Geopend: wo juni/aug/sept 12.00-16.00, of op a...","Open: Wed June/Aug/Sept 12:00-16:00, or by app...",53.066230,4.785018
6,Gallery-route1,02 ATELIER 'ET WAD,02 ATELIER 'ET WAD,"Pearlwerck 9, 1791 WB Den Burg, Texel",https://www.atelieretwadtexel.nl,"Geopend: Jaarrond wo-vr 13.30-16.00, of op afs...","Open: Year-round Wed-Fri 13:30-16:00, or by ap...",NaN,NaN
7,Gallery-route1,03 MUSEUM GALERIE RAT,03 MUSEUM GALLERY RAT,"Burgwal 20, 1791 AJ Den Burg, Texel",https://www.texel.net/,"Geopend: Jaarrond wo-za 11.00-17.00, of op afs...","Open: Year-round Wed-Sat 11:00-17:00, or by ap...",53.055015,4.798288
8,Gallery-route1,04 ATELIERGALERIE DE GARAGE,04 ATELIER GALLERY DE GARAGE,"Element 16, 1791 DD Den Burg, Texel",https://www.texel.net/,Geopend: wo april-sept 11.00-16.00.,Open: Wed April-Sept 11:00-16:00.,NaN,NaN
9,Gallery-route1,05 INA VADER,05 INA VADER,"Warmoesstraat 52, 1791 CS Den Burg, Texel",https://www.inavader.nl,"Geopend: wo 11.00-16.00, of op afspraak.","Open: Wed 11:00-16:00, or by appointment.",53.053279,4.798945
